# 01 - Download Dependencies & Package Offline Bundle
**Environment**: Kaggle GPU or CPU with **Internet ON**.

This notebook downloads all python wheels (`.whl`) and bundles the repository code + harmful prompt dataset so you can run attacks in a completely **offline (air-gapped)** Kaggle session.

---

### Workflow
1. Run this notebook with **Internet ENABLED**.
2. When finished, find `offline_attack_bundle.zip` in `/kaggle/working`.
3. In Kaggle's right sidebar under **Output**, click the three dots $\rightarrow$ **New Dataset** (name it `offline-attack-bundle`).
4. Attach that dataset to the offline attack notebook (`02_offline_attack_eval.ipynb`).


## 1 - Prepare Directory Structure

In [ ]:
import os, sys, shutil, pathlib, subprocess

WORK = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path("./workspace")
BUNDLE_DIR = WORK / "offline_attack_bundle"
WHEELS_DIR = BUNDLE_DIR / "wheels"
REPO_DIR = BUNDLE_DIR / "repo"

if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)

WHEELS_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR.mkdir(parents=True, exist_ok=True)
print(f"Bundle directory ready at: {BUNDLE_DIR}")


## 2 - Download Python Wheels for Offline Installation

In [ ]:
# Download wheels compatible with Python 3.10 / 3.11 Linux x86_64
packages = [
    "transformers>=4.45.0",
    "accelerate>=0.26.0",
    "sentencepiece",
    "protobuf",
    "tomli",
]

cmd = [
    sys.executable, "-m", "pip", "download",
    "-d", str(WHEELS_DIR),
    *packages
]
print("Downloading wheels... this may take 1-2 minutes.")
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    print("Download stderr:", res.stderr)
    raise RuntimeError("pip download failed")

downloaded_wheels = list(WHEELS_DIR.glob("*.whl"))
print(f"Successfully downloaded {len(downloaded_wheels)} wheels to {WHEELS_DIR}")


## 3 - Bundle Repository Code & Dataset

In [ ]:
# Determine repo location (either current directory or clone if needed)
current_path = pathlib.Path.cwd()
if (current_path / "run_eval.py").exists():
    src_repo = current_path
elif (current_path / "repo" / "run_eval.py").exists():
    src_repo = current_path / "repo"
else:
    # Clone from GitHub
    print("Cloning repository from GitHub...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/MehemudAzad/LLM-jailbreaking-with-layered-prompt-defense.git",
                    str(src_repo := WORK / "temp_clone")], check=True)

# Copy source tree components into bundle
for item in ["attacks", "core", "defense", "datasets", "config.toml", "run_eval.py", "report.py"]:
    src = src_repo / item
    dst = REPO_DIR / item
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    elif src.is_file():
        shutil.copy2(src, dst)

# Ensure harmful dataset is generated and included
harmful_gen = REPO_DIR / "datasets" / "build_harmful.py"
if harmful_gen.exists():
    subprocess.run([sys.executable, str(harmful_gen)], cwd=str(REPO_DIR), check=True)
    print("Harmful behaviors dataset generated at repo/datasets/harmful_behaviors.jsonl")

print("Repository code successfully bundled.")


## 4 - Create Offline Zip Archive

In [ ]:
archive_path = WORK / "offline_attack_bundle.zip"
if archive_path.exists():
    archive_path.unlink()

print("Creating zip archive...")
shutil.make_archive(str(WORK / "offline_attack_bundle"), "zip", root_dir=str(BUNDLE_DIR))
size_mb = archive_path.stat().st_size / (1024 * 1024)
print(f"Created {archive_path.name} ({size_mb:.1f} MB)")


## 5 - Sanity Check: Test Model Loading (Gemma)

In [ ]:
# Validate that the downloaded wheels and the local Gemma model work end-to-end
import os, sys, glob, json, pathlib, subprocess

# 1. Test installing the downloaded wheels locally
print("Testing wheel installation...")
res = subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-index", f"--find-links={WHEELS_DIR}",
    "transformers", "accelerate", "sentencepiece", "protobuf"
], capture_output=True, text=True)
print("Wheel install output:", res.stdout[-200:] if res.stdout else res.stderr[-200:])

# 2. Check model path
TEST_PATH = pathlib.Path("/kaggle/input/models/google/gemma/pytorch/2b-it/2")

# If the path differs slightly, attempt to auto-discover
if not TEST_PATH.exists():
    for candidate in pathlib.Path("/kaggle/input").rglob("*2b-it*"):
        if candidate.is_dir() and ((candidate / "config.json").exists() or (candidate / "model.safetensors.index.json").exists()):
            TEST_PATH = candidate
            break

print(f"Target model location: {TEST_PATH}")
if not TEST_PATH.exists():
    print("⚠️ Model path not found in /kaggle/input! If you haven't attached Gemma to this session, attach it in the right sidebar under 'Models' or 'Input'.")
else:
    # 3. Add bundled repo to sys.path and run a 1-goal test
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    import run_eval
    from core.config import CONFIG
    CONFIG["models"]["target"]["name"] = str(TEST_PATH)
    CONFIG["models"]["target"]["revision"] = None
    CONFIG["models"]["target"]["device"] = "auto"
    CONFIG["models"]["target"]["max_memory"] = None

    cmd_args = [
        "--attack", "passthrough,prefix_injection",
        "--defense", "off",
        "--no-grade",
        "--limit", "1",
        "--tag", "gemma-sanity",
    ]

    print("\nRunning a 1-goal test (passthrough & prefix_injection) with defense=off, no-grade...")
    run_eval.main(cmd_args)

    # 4. Read newest transcript to verify completions
    logs = sorted(glob.glob(str(REPO_DIR / "logs/*/transcript.jsonl")), key=os.path.getmtime)
    if logs:
        print("\n" + "=" * 70)
        print("SANITY CHECK RESULTS:")
        print("=" * 70)
        with open(logs[-1], "r", encoding="utf-8") as f:
            for line in f:
                d = json.loads(line)
                if d.get("type") == "trial":
                    print(f"Attack   : {d.get('attack')}")
                    print(f"Prompt   : {(d.get('prompt') or '')[:70]}...")
                    print(f"Response : {(d.get('response') or '')[:150]}...")
                    print("-" * 70)
        print("\n✅ Model loaded and generated responses successfully!")
        print("You can now safely create the Kaggle dataset from 'offline_attack_bundle.zip'!")
